# 02 · Bonus: Ein Mini-Sprachmodell in zwei Minuten

In Notebook 01 hat ein Netz gelernt, Punkte zu trennen. Jetzt lernt ein Netz mit **derselben Trainingsschleife**, Text zu schreiben.

Der Trick, auf dem *jedes* Sprachmodell beruht – auch ChatGPT, Copilot und Claude:

> **Schau dir die letzten Zeichen an und rate, welches Zeichen als Nächstes kommt.**

Das ist alles. Wer das gut genug kann, kann scheinbar „schreiben". Unser Modell macht das mit einzelnen Buchstaben und einem winzigen Text – deshalb wird das Ergebnis lustig, nicht klug. Aber du siehst das Prinzip von innen.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)

text = open("texte/azubi_chroniken.txt", encoding="utf-8").read()
print(f"Der Trainingstext hat {len(text):,} Zeichen. So fängt er an:\n")
print(text[:300])

## 1 · Text wird zu Zahlen

Ein Netz rechnet nur mit Zahlen. Also bekommt jedes Zeichen (Buchstabe, Leerzeichen, Punkt …) eine Nummer. Große Sprachmodelle machen das mit Wortstücken („Token") statt mit Buchstaben – das Prinzip ist gleich.

In [ ]:
zeichen = sorted(set(text))
zeichen_zu_zahl = {z: i for i, z in enumerate(zeichen)}
zahl_zu_zeichen = {i: z for i, z in enumerate(zeichen)}

print(f"{len(zeichen)} verschiedene Zeichen: {''.join(zeichen)!r}\n")
beispiel = "Kaffee"
print(f"{beispiel!r} → {[zeichen_zu_zahl[z] for z in beispiel]}")

daten = torch.tensor([zeichen_zu_zahl[z] for z in text])

## 2 · Die Trainingsbeispiele

Das Modell darf immer die letzten **8 Zeichen** sehen (den *Kontext*) und soll das nächste vorhersagen. Aus dem Text schneiden wir uns dafür tausende solcher Beispiele.

In [ ]:
KONTEXT = 8

X = torch.stack([daten[i:i + KONTEXT] for i in range(len(daten) - KONTEXT)])
y = daten[KONTEXT:]
print(f"{len(X):,} Trainingsbeispiele. Die ersten fünf:\n")
for eingabe, ziel in zip(X[:5], y[:5]):
    kontext = ''.join(zahl_zu_zeichen[int(i)] for i in eingabe)
    print(f"  {kontext!r:14} →  {zahl_zu_zeichen[int(ziel)]!r}")

## 3 · Das Modell

Fast wie in Notebook 01 – nur der Eingang ist anders:

1. **Embedding:** jedes der 8 Zeichen wird zu einem kleinen Zahlenvektor (den das Modell selbst lernt)
2. Die 8 Vektoren werden aneinandergehängt
3. **Versteckte Schicht** mit `ReLU` – genau wie vorhin
4. **Ausgang:** eine Zahl pro möglichem Zeichen – je größer, desto wahrscheinlicher kommt es als Nächstes

In [ ]:
class MiniSprachmodell(nn.Module):
    def __init__(self, n_zeichen, kontext, embedding=24, hidden=256):
        super().__init__()
        self.embedding = nn.Embedding(n_zeichen, embedding)
        self.netz = nn.Sequential(
            nn.Linear(kontext * embedding, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_zeichen),
        )

    def forward(self, x):
        e = self.embedding(x)              # (Batch, 8, embedding)
        e = e.flatten(start_dim=1)         # (Batch, 8·embedding)
        return self.netz(e)                # (Batch, n_zeichen)

model = MiniSprachmodell(len(zeichen), KONTEXT)
print(f"{sum(p.numel() for p in model.parameters()):,} Gewichte")

## 4 · Training – dieselbe Schleife wie in Notebook 01

Vorhersage → Fehler messen → Schuld verteilen → Schritt gehen. Neu ist nur, dass wir pro Schritt eine zufällige **Portion** (Batch) von 128 Beispielen nehmen statt alle auf einmal – so machen es auch die Großen.

In [ ]:
SCHRITTE = 3000
BATCH    = 128

optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
start = time.time()
verlauf = []

for schritt in range(SCHRITTE):
    idx = torch.randint(0, len(X), (BATCH,))
    logits = model(X[idx])                   # 1. Vorhersage
    loss = F.cross_entropy(logits, y[idx])   # 2. Fehler messen
    optimizer.zero_grad()
    loss.backward()                          # 3. Schuld verteilen
    optimizer.step()                         # 4. Schritt gehen
    verlauf.append(loss.item())
    if schritt % 500 == 0:
        print(f"Schritt {schritt:4d}   Loss {loss.item():.2f}")

print(f"\nFertig nach {time.time() - start:.0f} Sekunden.")
plt.figure(figsize=(6, 3)); plt.plot(verlauf); plt.xlabel("Schritt"); plt.ylabel("Loss"); plt.grid(alpha=.3);

## 5 · Das Modell schreibt

Jetzt lassen wir es Zeichen für Zeichen weiterschreiben: Kontext rein → Wahrscheinlichkeiten raus → ein Zeichen ziehen → anhängen → von vorn.

Die **Temperatur** steuert, wie mutig gezogen wird: niedrig = immer das wahrscheinlichste Zeichen (langweilig, wiederholt sich), hoch = auch unwahrscheinliche Zeichen (kreativ bis wirr).

In [ ]:
def schreibe(anfang, laenge=250, temperatur=0.8):
    anfang = anfang.rjust(KONTEXT)                 # links auffüllen
    kontext = [zeichen_zu_zahl.get(z, 0) for z in anfang[-KONTEXT:]]
    ergebnis = anfang.lstrip()
    with torch.no_grad():
        for _ in range(laenge):
            logits = model(torch.tensor([kontext]))
            p = F.softmax(logits / temperatur, dim=-1)
            naechstes = torch.multinomial(p, 1).item()
            ergebnis += zahl_zu_zeichen[naechstes]
            kontext = kontext[1:] + [naechstes]
    return ergebnis

print(schreibe("Montag. ", temperatur=0.8))

Fällt dir auf, dass ganze Sätze **wörtlich** aus dem Trainingstext kommen? Der Text ist winzig (6 KB) und das Modell hat 67.000 Gewichte – es hat den Text weitgehend **auswendig gelernt** und mischt ihn an den Nahtstellen neu. Das ist genau das Overfitting aus Notebook 01, nur diesmal mit Text. Große Sprachmodelle haben dasselbe Problem: Sie können Trainingstexte wortwörtlich wiedergeben, ohne sie zu „verstehen".

## 6 · 🔧 Drehknöpfe

- **Temperatur:** `0.2` / `0.8` / `1.5` – lies dir die drei Texte laut vor.
- **Anfang:** `"Die Ausbilderin sagt: "` oder `"Das Ticket "` – erkennt man, was das Modell „gelernt" hat?
- **Kontext:** oben `KONTEXT = 3` setzen und alles ab Abschnitt 2 nochmal laufen lassen – wie viel schlechter wird es mit weniger Gedächtnis?

In [ ]:
for temperatur in [0.2, 0.8, 1.5]:
    print(f"── Temperatur {temperatur} ──────────────────────────────")
    print(schreibe("Die Ausbilderin sagt: ", laenge=200, temperatur=temperatur))
    print()

## 7 · Was „denkt" das Modell?

Ein Sprachmodell antwortet nie mit *einem* Zeichen – es liefert für **jedes** mögliche Zeichen eine Wahrscheinlichkeit. Hier die Top 8 nach einem Kontext:

In [ ]:
def zeige_kandidaten(kontext_text, top=8):
    kontext = [zeichen_zu_zahl.get(z, 0) for z in kontext_text.rjust(KONTEXT)[-KONTEXT:]]
    with torch.no_grad():
        p = F.softmax(model(torch.tensor([kontext])), dim=-1)[0]
    werte, indizes = p.topk(top)
    labels = [repr(zahl_zu_zeichen[int(i)]) for i in indizes]
    plt.figure(figsize=(7, 3))
    plt.bar(labels, werte.numpy() * 100, color="#E8801C")
    plt.ylabel("%")
    plt.title(f"Nach {kontext_text!r} kommt als Nächstes …")
    plt.show()

zeige_kandidaten("Die ")        # hier ist das Modell unsicher
zeige_kandidaten("Der Kaff")    # hier ist es sich sehr sicher

Genau so „weiß" ChatGPT, dass nach *„Die Hauptstadt von Bayern ist"* mit hoher Wahrscheinlichkeit *„München"* kommt – nicht, weil es Geografie versteht, sondern weil diese Zeichenfolge in den Trainingsdaten sehr oft so weiterging. **Und genauso entstehen Halluzinationen:** Wenn die Daten nichts hergeben, wird trotzdem das wahrscheinlichste Zeichen gezogen – selbstbewusst und falsch.

## 8 · 🔧 Eigener Text

Ersetze den Inhalt von `texte/azubi_chroniken.txt` durch einen anderen Text – zum Beispiel lässt du dir von Copilot 3000 Wörter über dein Lieblingsthema schreiben und fügst sie ein. Dann alles von oben neu ausführen. **Mehr Text = besseres Modell**, das gilt hier genauso wie bei den Großen.

## 9 · Von hier zu ChatGPT

| | Unser Mini-Modell | Echtes Sprachmodell |
|---|---|---|
| Einheit | Buchstabe | Token (Wortstück) |
| Gedächtnis (Kontext) | 8 Zeichen | 100.000+ Token |
| Architektur | Embedding + eine Schicht | Transformer mit ~100 Schichten |
| Gewichte | ~70.000 | Hunderte Milliarden |
| Trainingstext | 6 KB Azubi-Chroniken | ein großer Teil des Internets |
| Danach | fertig | Feinschliff durch menschliches Feedback, damit es antwortet statt nur weiterzuschreiben |

Der Kern – **das nächste Stück raten, aus dem Fehler lernen, wiederholen** – ist identisch. Wenn du das verstanden hast, verstehst du mehr über KI als die meisten, die täglich damit arbeiten.